# 9. Полный Transformer — Encoder-Decoder

**Цель:** Собрать полную seq2seq архитектуру трансформера: энкодер, декодер, эмбеддинги, выходную голову. Продемонстрировать авторегрессивную генерацию на задаче копирования.

---

In [2]:
import sys, os, logging, math

import torch  # Основной фреймворк
import torch.nn as nn  # Слои
import torch.nn.functional as F  # Функции
import numpy as np  # Численные расчёты
import matplotlib.pyplot as plt  # Графики

if torch.cuda.is_available():  # GPU NVIDIA
    device = torch.device("cuda")

elif torch.backends.mps.is_available():  # GPU Apple
    device = torch.device("mps")

else:  # CPU
    device = torch.device("cpu")

## 9.1 Компоненты трансформера

Переиспользуем все реализации из предыдущих ноутбуков в одном классе.

In [4]:

class MultiHeadAttention(nn.Module):  # Многоголовое внимание
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0  # d_model кратен n_heads
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Размерность головы
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # Query
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # Key
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # Value
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # Output
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        Q = self.W_Q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, n_heads, seq_len, seq_len)
        if mask is not None:  # Применяем маску (padding или causal)
            scores = scores.masked_fill(mask == 0, float('-inf'))  # Маскированные позиции → -inf
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.d_model)  # Собираем головы обратно
        return self.W_O(output)  # Выходная проекция

class FeedForward(nn.Module):  # Двухслойная FFN
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # По умолчанию 4×d_model
        self.fc1 = nn.Linear(d_model, d_ff)  # Расширение d_model → d_ff
        self.fc2 = nn.Linear(d_ff, d_model)  # Сжатие d_ff → d_model
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))  # Linear → GELU → Dropout → Linear

class EncoderBlock(nn.Module):  # Блок энкодера: Self-Attention → Add&Norm → FFN → Add&Norm
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)  # Self-attention
        self.ffn = FeedForward(d_model, d_ff, dropout)  # Feed-Forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):  # Прямой проход энкодера
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))  # Self-attention + Add&Norm
        x = x + self.dropout2(self.ffn(self.norm2(x)))  # FFN + Add&Norm
        return x

class DecoderBlock(nn.Module):  # Блок декодера: три под-слоя
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)  # 1. Masked Self-Attention
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)  # 2. Cross-Attention
        self.ffn = FeedForward(d_model, d_ff, dropout)  # Feed-Forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):  # Прямой проход декодера
        # Causal self-attention (ручная маскировка)
        seq_len = x.size(1)
        causal = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()  # Каузальная маска
        combined_mask = tgt_mask.unsqueeze(1).unsqueeze(2) if tgt_mask is not None else None  # Комбинируем padding + causal
        if combined_mask is not None:
            combined_mask = combined_mask | causal.unsqueeze(0).unsqueeze(0)
        else:
            combined_mask = causal.unsqueeze(0).unsqueeze(0)
        x = x + self.dropout1(self.self_attention(self.norm1(x), self.norm1(x), self.norm1(x), combined_mask))  # Masked Self-Attention
        x = x + self.dropout2(self.cross_attention(self.norm2(x), self.norm2(encoder_output), self.norm2(encoder_output), src_mask))  # Cross-Attention
        x = x + self.dropout3(self.ffn(self.norm3(x)))  # FFN
        return x

## 9.2 PositionalEncoding

In [6]:
class PositionalEncoding(nn.Module):  # Синусоидальные PE
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))  # Частоты
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])  # Добавляем PE + Dropout

## 9.3 Полный Transformer

Объединяем всё: эмбеддинги, PE, энкодер, декодер, output head.

In [8]:

class Transformer(nn.Module):  # Полный трансформер (Encoder-Decoder)
    def __init__(self, vocab_size, d_model, n_heads, num_encoder_layers,
                 num_decoder_layers, d_ff=None, max_len=5000, dropout=0.1):
        super().__init__()
        self.d_model = d_model  # Размерность модели
        
        # Embeddings + Positional Encoding
        self.embedding = nn.Embedding(vocab_size, d_model)  # Эмбеддинги токенов
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)  # Позиционные кодирования
        
        # Encoder
        self.encoder_layers = nn.ModuleList([  # Стек энкодера
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        
        # Decoder
        self.decoder_layers = nn.ModuleList([  # Стек декодера
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        
        # Output head
        self.output_proj = nn.Linear(d_model, vocab_size)  # Выходная проекция
        
        # Инициализация
        for p in self.parameters():  # Инициализация Xavier
            if p.dim() > 1:  # Только веса матриц (не bias)
                nn.init.xavier_uniform_(p)  # Равномерный Xavier
        
    
    def encode(self, src, src_mask=None):  # Кодирование исходной последовательности
        x = self.pos_encoding(self.embedding(src) * math.sqrt(self.d_model))  # Эмбеддинги + PE (с масштабированием)
        for layer in self.encoder_layers:  # Проход по слоям энкодера
            x = layer(x, src_mask)  # Прямой проход через блок
        return x
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):  # Декодирование целевой последовательности
        x = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.d_model))  # Эмбеддинги + PE для декодера
        for layer in self.decoder_layers:  # Проход по слоям декодера
            x = layer(x, encoder_output, src_mask, tgt_mask)  # Прямой проход через блок
        return x
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):  # Полный прямой проход
        enc_output = self.encode(src, src_mask)  # Шаг 1: кодируем источник
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)  # Шаг 2: декодируем с cross-attention
        return self.output_proj(dec_output)  # Шаг 3: проекция на словарь

In [9]:

vocab_size = 20  # Размер словаря
model = Transformer(  # Создаём трансформер
    vocab_size=vocab_size,
    d_model=32,
    n_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=64,
    max_len=50,
).to(device)

src = torch.randint(0, vocab_size, (4, 10)).to(device)  # (batch, src_len)
tgt = torch.randint(0, vocab_size, (4, 8)).to(device)   # (batch, tgt_len)

output = model(src, tgt)
print(f"Source shape:          {src.shape}")
print(f"Target shape:          {tgt.shape}")
print(f"Output shape:          {output.shape}")
print(f"Output dim:            {output.shape[-1]} (should be {vocab_size})")

Source shape:          torch.Size([4, 10])
Target shape:          torch.Size([4, 8])
Output shape:          torch.Size([4, 8, 20])
Output dim:            20 (should be 20)


## 9.4 Демонстрация: задача копирования

Обучим трансформер копировать последовательности.

In [11]:

from torch.utils.data import DataLoader, TensorDataset  # Загрузчик данных

BOS, EOS, PAD = 0, 1, 2  # Специальные токены

def generate_copy_data(num_samples, max_len, vocab_size):  # Генерация пар src-tgt для копирования
    """Генерирует пары src-tgt для задачи копирования."""
    src_list, tgt_list = [], []
    for _ in range(num_samples):
        length = np.random.randint(2, max_len + 1)  # Случайная длина последовательности
        seq = np.random.randint(3, vocab_size, size=length).tolist()
        src = [BOS] + seq + [EOS]  # Исходная: [BOS, ..., EOS]
        src = src + [PAD] * (max_len + 2 - len(src))  # Дополняем до фиксированной длины
        
        tgt_in = [BOS] + seq + [EOS]  # Вход декодера
        tgt_in = tgt_in + [PAD] * (max_len + 2 - len(tgt_in))
        
        tgt_out = seq + [EOS]  # Целевая последовательность (сдвиг на 1)
        tgt_out = tgt_out + [PAD] * (max_len + 2 - len(tgt_out))
        
        src_list.append(src)
        tgt_list.append(tgt_out)
    
    return (torch.tensor(src_list), torch.tensor(tgt_list))

vocab_size = 16
max_len = 6
train_src, train_tgt = generate_copy_data(500, max_len, vocab_size)  # 500 обучающих примеров
print(f"Train src shape: {train_src.shape}")
print(f"Train tgt shape: {train_tgt.shape}")
print(f"Source example:   {train_src[0].tolist()}")
print(f"Target example:   {train_tgt[0].tolist()}")

Train src shape: torch.Size([500, 8])
Train tgt shape: torch.Size([500, 8])
Source example:   [0, 11, 4, 8, 1, 2, 2, 2]
Target example:   [11, 4, 8, 1, 2, 2, 2, 2]


In [12]:

model_small = Transformer(  # Маленький трансформер для копирования
    vocab_size=vocab_size,
    d_model=32,
    n_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=64,
    max_len=50,
    dropout=0.1,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # Функция потерь (игнорируем PAD)
optimizer = torch.optim.Adam(model_small.parameters(), lr=0.001)  # Оптимизатор

n_epochs = 50  # Количество эпох
batch_size = 32  # Размер мини-батча
losses = []  # Список для логирования потерь

for epoch in range(n_epochs):  # Цикл обучения
    epoch_loss = 0
    n_batches = 0
    perm = torch.randperm(len(train_src))  # Перемешивание на каждую эпоху
    
    for i in range(0, len(train_src), batch_size):
        idx = perm[i:i+batch_size]
        src = train_src[idx].to(device)
        tgt = train_tgt[idx].to(device)
        
        # Teacher forcing: decoder input = BOS + target[:-1]
        dec_input = torch.cat([
            torch.full((len(idx), 1), BOS, device=device, dtype=torch.long),
            tgt[:, :-1]
        ], dim=1)
        
        output = model_small(src, dec_input)
        loss = criterion(output.reshape(-1, vocab_size), tgt.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_small.parameters(), 1.0)  # Клиппинг градиентов
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: loss={avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: Copy Task')
plt.grid(True)
plt.show()

## 9.5 Авторегрессивная генерация

Генерируем последовательность по одному токену.

In [14]:

@torch.no_grad()  # Отключаем вычисление градиентов
def generate(model, src, max_len=10):  # Авторегрессивная генерация
    model.eval()  # Режим инференса
    src = src.to(device)
    enc_output = model.encode(src)  # Кодируем источник
    
    # Начинаем с BOS
    tgt = torch.full((src.size(0), 1), BOS, dtype=torch.long, device=device)  # Начинаем с BOS
    
    for i in range(max_len):  # Генерируем до max_len токенов
        dec_output = model.decode(tgt, enc_output)  # Декодируем всю последовательность
        logits = model.output_proj(dec_output[:, -1:, :])  # Логиты только последнего токена
        next_token = logits.argmax(dim=-1)  # Жадное декодирование (argmax)
        tgt = torch.cat([tgt, next_token], dim=1)  # Добавляем токен к последовательности
        
        # Стоп при EOS
        if (next_token == EOS).all():  # Останавливаемся при генерации EOS
            break
    
    return tgt

test_src, _ = generate_copy_data(4, 4, vocab_size)  # Тестовые примеры
for i, src in enumerate(test_src[:3]):  # Проверяем 3 примера
    src = src.unsqueeze(0).to(device)
    pred = generate(model_small, src, max_len=12)
    src_tokens = [t for t in src[0].tolist() if t not in (BOS, EOS, PAD)]
    pred_tokens = [t for t in pred[0].tolist() if t not in (BOS, EOS, PAD)]
    print(f"Sample {i+1}: src={src_tokens}, pred={pred_tokens}, match={src_tokens == pred_tokens}")

RuntimeError: The size of tensor a (9) must match the size of tensor b (8) at non-singleton dimension 1

In [15]:
print("=== Full Transformer complete ===")
print("Topics covered:")
print("  - Full Transformer: encoder + decoder + output head")
print("  - Tokenization with special tokens (BOS, EOS, PAD)")
print("  - Embeddings + PositionalEncoding")
print("  - Copy task: training with teacher forcing")
print("  - Autoregressive generation (greedy decoding)")
print(f"  - Final loss: {losses[-1]:.4f}")

=== Full Transformer complete ===
Topics covered:
  - Full Transformer: encoder + decoder + output head
  - Tokenization with special tokens (BOS, EOS, PAD)
  - Embeddings + PositionalEncoding
  - Copy task: training with teacher forcing
  - Autoregressive generation (greedy decoding)
  - Final loss: nan
